# DocxParser Demo

Shows flat mode and tree mode extraction on `tests/fixtures/sample.docx`.


| Aspect | Notes |
|--------|-------|
| **Pros** | Deterministic heading detection via Word `Heading N` styles — no font-size guessing; reliable image and table extraction; lightweight (python-docx only) |
| **Cons** | Requires the author to apply Word heading styles — decoratively formatted paragraphs are not detected as headings; no page number support (DOCX has no fixed page boundaries) |
| **vs. PdfParser** | DOCX heading detection is always accurate (style-based) vs. PDF's heuristic font-size ranking; DOCX cannot be scanned/image-only so all text is always extractable |
| **vs. unstructured.io** | unstructured handles more edge cases (e.g. complex tables, embedded objects) but adds heavy dependencies; DocxParser covers the common case with a single lightweight library |

## Imports

In [1]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.parsers.factory import ParserFactory

## Fixture

In [2]:
FIXTURES = pathlib.Path.cwd().parent.parent / "tests" / "fixtures"
DOCX_PATH = str(FIXTURES / "sample.docx")

if not (FIXTURES / "sample.docx").exists():
    from tests.fixtures.docx import create_sample_docx
    create_sample_docx(FIXTURES / "sample.docx")

print("DOCX:", DOCX_PATH)

DOCX: c:\Users\SidNa\Documents\GitHub\AI\cleave\notebooks\tests\fixtures\sample.docx


| Mode | `Document` field | Use when |
|------|-----------------|----------|
| `flat` | `.pages` — one virtual `DocumentPage` (page_number=None) | Simple text retrieval, fixed chunking |
| `tree` | `.root` — recursive `TreeNode` hierarchy | Semantic / layout-aware chunking |

## Flat mode

In [3]:
flat_doc_parser = ParserFactory.create(DOCX_PATH, mode="flat")
flat_doc = flat_doc_parser.parse()

# DOCX has no page concept — one virtual page with page_number=None
print(f"Pages      : {len(flat_doc.pages)}")
print(f"Page num   : {flat_doc.pages[0].page_number}")  # None
print(f"Root       : {flat_doc.root}")  # None

Pages      : 1
Page num   : None
Root       : None


In [4]:
page = flat_doc.pages[0]
print(f"{len(page.blocks)} blocks total\n")
for block in page.blocks:
    preview = block.content[:100].replace("\n", " ")
    print(f"[{block.type.value:5}]  {preview}")

9 blocks total

[text ]  Sample Document
[text ]  Introduction
[text ]  This is the introduction paragraph.
[text ]  Data Overview
[text ]  This section covers data.
[image]  iVBORw0KGgoAAAANSUhEUgAAABQAAAAUCAIAAAAC64paAAAAG0lEQVR4nGP4z8BANiJf56jmUc2jmkc1U0UzADHNjoAymaoJAAAA
[table]  | Name | Value | | --- | --- | | Alpha | 1 | | Beta | 2 |
[text ]  Conclusion
[text ]  Final remarks go here.


In [5]:
print(f"Tables : {len(flat_doc.all_tables)}")
print(f"Images : {len(flat_doc.all_images)}")

if flat_doc.all_tables:
    print("\nFirst table (Markdown):")
    print(flat_doc.all_tables[0].content)

Tables : 1
Images : 1

First table (Markdown):
| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |


## Tree mode

In [6]:
tree_doc_parser = ParserFactory.create(DOCX_PATH, mode="tree")
tree_doc = tree_doc_parser.parse()

print(f"Pages         : {tree_doc.pages}")  # None
print(f"Root children : {len(tree_doc.root.children)}")

Pages         : None
Root children : 1


In [7]:
def print_tree(node, indent=0):
    role  = node.metadata.get("role", "")
    level = node.metadata.get("level", "")
    label = f"[{node.content_type.value}]"
    if role:  label += f" role={role}"
    if level: label += f" level={level}"
    preview = node.content[:70].replace("\n", " ") if node.content else ""
    print("  " * indent + f"{label}  \"{preview}\"")
    for child in node.children:
        print_tree(child, indent + 1)

print_tree(tree_doc.root)

[text] role=root  ""
  [text] role=heading level=1  "Sample Document"
    [text] role=heading level=2  "Introduction"
      [text] role=paragraph  "This is the introduction paragraph."
    [text] role=heading level=2  "Data Overview"
      [text] role=paragraph  "This section covers data."
      [image]  "iVBORw0KGgoAAAANSUhEUgAAABQAAAAUCAIAAAAC64paAAAAG0lEQVR4nGP4z8BANiJf56"
      [table]  "| Name | Value | | --- | --- | | Alpha | 1 | | Beta | 2 |"
    [text] role=heading level=2  "Conclusion"
      [text] role=paragraph  "Final remarks go here."


### Heading hierarchy
Because DOCX uses explicit styles, the heading levels match the Word document exactly.

In [8]:
def collect_headings(node):
    headings = []
    if node.metadata.get("role") == "heading":
        headings.append((node.metadata["level"], node.content))
    for child in node.children:
        headings.extend(collect_headings(child))
    return headings

print(f"{'Level':>5}  Text")
print("-" * 40)
for level, text in collect_headings(tree_doc.root):
    print(f"H{level:>4}  {text}")

Level  Text
----------------------------------------
H   1  Sample Document
H   2  Introduction
H   2  Data Overview
H   2  Conclusion


In [9]:
print("=== FULL TEXT ===")
print(tree_doc.full_text[:400])
print("\n=== TREE TO MARKDOWN (FOR TEXT STYLE CHUNKING) ===")
print(tree_doc.to_markdown()[:400])

=== FULL TEXT ===
Sample Document
Introduction
This is the introduction paragraph.
Data Overview
This section covers data.
| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |
Conclusion
Final remarks go here.

=== TREE TO MARKDOWN (FOR TEXT STYLE CHUNKING) ===
# Sample Document

## Introduction

This is the introduction paragraph.

## Data Overview

This section covers data.

![image]()

| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |

## Conclusion

Final remarks go here.


## Summary comparison

In [10]:
def count_nodes(node):
    return 1 + sum(count_nodes(c) for c in node.children)

print(f"{'Mode':<6}  {'Pages':>5}  {'Root nodes':>10}  {'Tables':>6}  {'Images':>6}")
print("-" * 40)
print(f"{'flat':<6}  {len(flat_doc.pages):>5}  {'N/A':>10}  {len(flat_doc.all_tables):>6}  {len(flat_doc.all_images):>6}")
print(f"{'tree':<6}  {'N/A':>5}  {count_nodes(tree_doc.root):>10}  {len(tree_doc.all_tables):>6}  {len(tree_doc.all_images):>6}")

Mode    Pages  Root nodes  Tables  Images
----------------------------------------
flat        1         N/A       1       1
tree      N/A          10       1       1
